# 🏦 AML Transaction Monitoring Model
**Author:** Anushka Shinde | MS Finance, Boston University  
**Stage 1:** Setup & Data Generation

---
In this notebook, we will build an Anti-Money Laundering (AML) transaction monitoring system from scratch.  
By the end of all stages, you will have:
- A dataset of 500 realistic banking transactions
- Rule-based red flags (structuring, high-risk countries, shell companies, etc.)
- A Machine Learning model that scores each transaction for risk
- A formatted Excel report ready to show employers

**This is Stage 1 — we focus only on setting up and generating the data.**

## Step 1: Import Libraries

Libraries are pre-built toolkits that save us from writing everything from scratch.  
In Google Colab, `pandas` and `numpy` are already installed — we just import them.

In [ ]:
import pandas as pd        # For working with data tables (like Excel in Python)
import numpy as np         # For math and generating random numbers

print('✅ Libraries loaded successfully!')

## Step 2: Set the Random Seed

We use a random seed so that every time you run this notebook, you get the **exact same data**.  
This is called **reproducibility** — very important in financial analysis and auditing.

In [ ]:
np.random.seed(42)

print('✅ Random seed set to 42.')
print('   This means our data will be identical every time we run the notebook.')

## Step 3: Define the Categories

Before generating transactions, we define what types of values can appear in our dataset.  
In a real bank, this data would come from their core banking system or a data warehouse.

> **What is FATF?**  
> The Financial Action Task Force — the global body that sets AML/CFT standards and publishes lists of high-risk jurisdictions.

In [ ]:
customer_types = ['Individual', 'Business', 'Shell Company', 'NGO']

countries = [
    'USA', 'UK', 'India', 'Cayman Islands', 'Panama',
    'Switzerland', 'Germany', 'UAE', 'Nigeria', 'Singapore'
]

# High-risk countries flagged by FATF
high_risk_countries = ['Cayman Islands', 'Panama', 'Nigeria']

transaction_types = [
    'Wire Transfer', 'Cash Deposit', 'ATM Withdrawal',
    'Online Transfer', 'Check', 'Crypto Exchange'
]

print('✅ Categories defined.')
print(f'   Total countries     : {len(countries)}')
print(f'   High-risk countries : {high_risk_countries}')
print(f'   Transaction types   : {transaction_types}')

## Step 4: Generate the Transaction Dataset

We now create 500 synthetic transactions. Each row represents one transaction.

Pay attention to the **Transaction Amount** logic:  
- 95% of transactions are normal (random amounts from an exponential distribution)  
- 5% are between **\$9,000–\$9,999** — this is called **structuring**, a classic money laundering technique where people break up amounts to stay just below the \$10,000 reporting threshold (CTR — Currency Transaction Report)

> **Why \$10,000?**  
> In the USA, banks are legally required to file a Currency Transaction Report (CTR) for any cash transaction above \$10,000. Money launderers deliberately stay below this to avoid detection.

In [ ]:
n = 500  # Number of transactions

df = pd.DataFrame({

    # Unique ID for each transaction
    'Transaction_ID': [f'TXN{str(i).zfill(5)}' for i in range(1, n + 1)],

    # Customer ID — some customers repeat (realistic)
    'Customer_ID': [f'CUST{np.random.randint(1000, 2000)}' for _ in range(n)],

    # Type of customer
    'Customer_Type': np.random.choice(
        customer_types, n,
        p=[0.5, 0.3, 0.1, 0.1]  # 50% Individual, 30% Business, 10% Shell, 10% NGO
    ),

    # Transaction Amount (with 5% structuring pattern built in)
    'Transaction_Amount': np.round(
        np.where(
            np.random.rand(n) > 0.95,
            np.random.uniform(9000, 9999, n),                          # Suspicious
            np.random.exponential(scale=3000, size=n).clip(100, 100000) # Normal
        ), 2),

    # Type of transaction
    'Transaction_Type': np.random.choice(transaction_types, n),

    # Where the money is coming FROM
    'Origin_Country': np.random.choice(
        countries, n,
        p=[0.3, 0.15, 0.15, 0.05, 0.05, 0.08, 0.1, 0.05, 0.04, 0.03]
    ),

    # Where the money is going TO
    'Destination_Country': np.random.choice(countries, n),

    # How many times this customer transacted in the last 30 days
    'Num_Transactions_Last_30Days': np.random.randint(1, 50, n),

    # This customer's average transaction over the last 6 months
    'Avg_Transaction_Last_6Months': np.round(
        np.random.exponential(scale=2000, size=n).clip(100, 50000), 2
    ),

    # How old is this account (in years)
    'Account_Age_Years': np.round(np.random.uniform(0.1, 20, n), 1),

    # Has a Suspicious Activity Report (SAR) been filed for this customer before?
    # SAR = report filed by banks to regulators (FinCEN in USA, FIU-IND in India)
    'Prior_SAR_Filed': np.random.choice([0, 1], n, p=[0.92, 0.08]),
})

print(f'✅ Dataset created with {len(df)} transactions and {len(df.columns)} columns!')

## Step 5: Preview the Data

Always preview your data after creating it to make sure it looks right.

In [ ]:
print('📋 First 5 rows of your dataset:')
df.head()

In [ ]:
print('📊 Dataset shape:', df.shape)
print('\nColumn names:')
for col in df.columns:
    print(f'  - {col}')

In [ ]:
print('💰 Transaction Amount Statistics:')
df['Transaction_Amount'].describe().round(2)

In [ ]:
print('👤 Customer Type Distribution:')
df['Customer_Type'].value_counts()

In [ ]:
print('🌍 Top Origin Countries:')
df['Origin_Country'].value_counts()

---
## ✅ Stage 1 Complete!

You have successfully built a dataset of 500 realistic banking transactions.

### Before moving to Stage 2, make sure you can answer these:

1. **Why did we set `np.random.seed(42)`?**
2. **Why are amounts between \$9,000–\$9,999 suspicious in AML?**
3. **What is a SAR and who do banks file it with?**
4. **What is FATF and why does it matter?**

> All answers are in the comments and markdown cells above — read through them carefully.

---
**Next → Stage 2: Rule-Based AML Red Flags**  
We'll write rules to flag suspicious transactions: structuring, high-risk countries, shell companies, smurfing, and more.